# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mvdu12/ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

url = 'https://raw.githubusercontent.com/Mvdu12/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric_feats = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
cat_feats = ['content_type', 'main_intent', 'competition_level']
X = df[numeric_feats + cat_feats].copy()
for c in numeric_feats:
    X[c + '_missing'] = X[c].isna().astype(int)
    X[c] = X[c].fillna(0)
X = pd.get_dummies(X, columns=cat_feats, dummy_na=True)
y = df['is_declining_label']

# Model quality (precision@K, grouped split) was already established honestly in
# ML-08/ML-09. For THIS shipped queue I refit on 100% of the data -- standard
# practice once a model is validated: holding back 20% forever in production
# wastes real signal, and the held-out numbers already on record are what future
# readers should trust for "how good is this," not this refit.
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X, y)
df['predicted_decline_probability'] = rf.predict_proba(X)[:, 1].round(4)

# Action tiers off the single score -- top 10% / next 20% / rest.
q90 = df['predicted_decline_probability'].quantile(0.90)
q70 = df['predicted_decline_probability'].quantile(0.70)
df['action_label'] = np.select(
    [df['predicted_decline_probability'] >= q90, df['predicted_decline_probability'] >= q70],
    ['refresh_now', 'review_soon'],
    default='monitor',
)

# Reason code: built from the two globally strongest permutation-importance
# features from ML-08 (impressions_90d, content_age_days), thresholded at their
# own 75th percentile. Rows that don't clear either threshold get an honest
# 'model_flagged' label rather than a made-up specific reason -- a Random
# Forest's real reasoning is a combination the model found, not a single
# human-readable rule, and pretending otherwise would overstate what this table
# can explain.
imp_q75 = df['impressions_90d'].quantile(0.75)
age_q75 = df['content_age_days'].quantile(0.75)

def reason(row):
    hi_vis = row['impressions_90d'] >= imp_q75
    aging = row['content_age_days'] >= age_q75
    if hi_vis and aging:
        return 'high_visibility_and_aging'
    if hi_vis:
        return 'high_visibility'
    if aging:
        return 'aging_content'
    return 'model_flagged'

df['reason_code'] = df.apply(reason, axis=1)
df['action_rank'] = df['predicted_decline_probability'].rank(method='first', ascending=False).astype(int)

queue = df.sort_values('action_rank')[[
    'content_id', 'client_id', 'action_rank', 'predicted_decline_probability',
    'action_label', 'reason_code', 'impressions_90d', 'content_age_days',
    'avg_position', 'days_since_last_update', 'content_type', 'main_intent',
]]

print("action_label distribution:")
print(df['action_label'].value_counts())
print("\nreason_code distribution within refresh_now (top tier):")
print(df[df['action_label'] == 'refresh_now']['reason_code'].value_counts())
print(
    "\nMost of the top tier lands on 'model_flagged' rather than a single clean "
    "driver -- an honest finding in itself. It means the Random Forest's edge "
    "over the Week-4 rule (ML-08) comes from combining several moderate signals, "
    "not one big lever, which matches the non-linear-structure read from ML-08's "
    "error analysis."
)
print("\nTop 10:")
print(queue.head(10).to_string(index=False))


action_label distribution:
action_label
monitor        20999
review_soon     5994
refresh_now     3007
Name: count, dtype: int64

reason_code distribution within refresh_now (top tier):
reason_code
model_flagged      2185
high_visibility     798
aging_content        24
Name: count, dtype: int64

Most of the top tier lands on 'model_flagged' rather than a single clean driver -- an honest finding in itself. It means the Random Forest's edge over the Week-4 rule (ML-08) comes from combining several moderate signals, not one big lever, which matches the non-linear-structure read from ML-08's error analysis.

Top 10:
          content_id         client_id  action_rank  predicted_decline_probability action_label   reason_code  impressions_90d  content_age_days  avg_position  days_since_last_update    content_type   main_intent
content_45f96356f84e client_624b60c58c            1                         0.7954  refresh_now model_flagged              126               104           7.3         

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Who uses this
A content team lead, as a weekly triage list — which pages to look at first, not a work order to execute blindly.

### What for
Prioritizing **REVIEW** effort. `refresh_now` means “look at this page this week,” not “this page is broken” or “edit this exact way.”

### Where it stops being valid

- This is a single cross-sectional snapshot (one 90-day window). It does not know about seasonality, upcoming algorithm changes, or planned site work.

- It was trained and validated on this portfolio's existing clients (grouped by `client_id` in ML-08/ML-09). A brand-new client with a very different content style is **out-of-distribution** — treat early recommendations for a new client as lower-confidence until enough of their own data exists.

- `is_declining_label` reflects impression trend only. A page can lose impressions for reasons this model can't see (a manual Google penalty, a site migration, a tracking outage) and can hold steady for reasons unrelated to content quality (branded/navigational demand).

- `precision@K` from ML-08/ML-09 (~0.48–0.58 depending on K) means roughly **4–6 in 10 flagged pages are genuinely declining** — useful for prioritization, but nowhere near a guarantee for any single row.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Before acting on any `refresh_now` row, a person should check:

1. **Trend direction:** The page's `trend_direction` isn't already `up` or `stable` — ML-08 found weak picks exactly this way (rank 2 and 9 in the Week-4 baseline's top 10 were both clear of the decline they were flagged for).

2. **Existing plans:** The page isn't already scheduled for a redesign, merge, or removal — a refresh would be wasted effort.

3. **Average position:** `avg_position` isn't deep (50+) — per the FlyRank research paper's position-tier findings, a page that far down usually needs a ranking/keyword fix, not a content refresh, and a refresh alone likely won't move it.

4. **Engagement rate:** `engagement_rate` isn't near zero alongside strong impressions — that combination (seen in ML-08's error analysis) can mean an instrumentation gap, not a real content problem, and refreshing won't fix a tracking bug.

### No-go list — never automate these from this table alone:

- **No auto-publishing content changes.** This ranks **WHAT to look at**, never **WHAT to write**.

- **No automated deletion, de-indexing, or client billing decisions** based on a low score — `monitor` means “lower priority this week,” not “worthless.”

- **No treating `predicted_decline_probability` as a probability of causal harm from inaction** — it's a ranking signal from one snapshot, not a forecast with a confidence interval.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [2]:
current_base_rate = df['is_declining_label'].mean()
current_action_mix = df['action_label'].value_counts(normalize=True).round(3)

print(f"Reference values logged at ship time (this run):")
print(f"  is_declining_label base rate: {current_base_rate:.3f}")
print(f"  action_label mix: {dict(current_action_mix)}")


Reference values logged at ship time (this run):
  is_declining_label base rate: 0.542
  action_label mix: {'monitor': np.float64(0.7), 'review_soon': np.float64(0.2), 'refresh_now': np.float64(0.1)}


### Retrain / re-check triggers

Any of these should prompt a rerun of **ML-08 through this notebook**, not just a fresh score:

1. **Base rate drift:** If `is_declining_label`'s rate moves more than ~5 percentage points away from the logged value above on a fresh data pull, the portfolio's underlying behavior has shifted enough that the model's calibration is suspect.

2. **Precision@K drop:** If a spot-check on a fresh 4–8 week sample shows `precision@500` meaningfully below the ~0.58 established in ML-08/ML-09, retrain rather than keep shipping the old model.

3. **Feature drift:** A large shift in the median of `impressions_90d` or `content_age_days` (the two strongest global drivers) signals that the training population no longer matches production traffic.

4. **New client onboarding:** Per Section 2's limits, a new client with no history yet should be flagged as **low-confidence**, not silently scored alongside established clients.

5. **Calendar backstop:** Re-run quarterly regardless of the above — content portfolios and search behavior both drift slowly in ways no single trigger catches immediately.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
import os

os.makedirs('work/outputs', exist_ok=True)

queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)

summary = pd.DataFrame({
    'metric': [
        'rows', 'base_rate_is_declining', 'refresh_now_count', 'review_soon_count',
        'monitor_count', 'q90_threshold', 'q70_threshold',
    ],
    'value': [
        len(df), round(current_base_rate, 4), int((df['action_label'] == 'refresh_now').sum()),
        int((df['action_label'] == 'review_soon').sum()), int((df['action_label'] == 'monitor').sum()),
        round(q90, 4), round(q70, 4),
    ],
})
summary.to_json('work/outputs/action_playbook_summary.json', orient='records', indent=2)

print(f"Wrote {len(queue):,} ranked rows to work/outputs/action_playbook_queue.csv")
print("Wrote work/outputs/action_playbook_summary.json")
print(summary.to_string(index=False))


Wrote 30,000 ranked rows to work/outputs/action_playbook_queue.csv
Wrote work/outputs/action_playbook_summary.json
                metric      value
                  rows 30000.0000
base_rate_is_declining     0.5421
     refresh_now_count  3007.0000
     review_soon_count  5994.0000
         monitor_count 20999.0000
         q90_threshold     0.7039
         q70_threshold     0.6561


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.